# TPI · Módulo 2 — Segmentación de núcleos
**PSIB 2026Q2 · Grupo 12** — Lautersztein, Reggio, Ballester
*Detección de leucemia linfoblástica aguda (LLA) en imágenes de frotis de sangre periférica*

El Módulo 1 mostró que el núcleo **no se distingue en V** (fondo y núcleo tienen V parecido), sino en **saturación y tono**. Acá comparamos dos formas de segmentarlo a partir de eso:

1. **Umbral en S** (Otsu) sobre el canal de saturación.
2. ***k*-means en el plano (a\*, b\*)** de CIE Lab (2 clusters: núcleo vs. resto), con el cluster de mayor croma como núcleo.

El dataset trae, para cada imagen, una **segmentación de referencia a nivel de célula completa** (fondo descartado, no núcleo aislado). La usamos como **región de interés (ROI)** para no confundir núcleo con eritrocitos o artefactos de fondo, y comparamos los dos métodos **entre sí** (no hay máscara de referencia específica del núcleo para calcular Dice contra una "verdad" externa).

La salida es la función `segmentar()` con el método elegido, que usa el Módulo 3 (extracción de descriptores). Las conclusiones, con los resultados sobre el dataset completo, están al final.

## 0. Carga
Mismo dataset e índice que en el Módulo 1 (ver ese notebook para el detalle de clases y columnas). Acá sólo necesitamos la imagen **original** y su **segmentación de referencia**, y la corrección de iluminación ya elegida (`cierre`).

In [ ]:
import os, glob
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import kagglehub

RUTA = kagglehub.dataset_download('mehradaria/leukemia')
df = pd.DataFrame({'ruta': sorted(glob.glob(f'{RUTA}/**/*.jp*g', recursive=True))})
df['tipo'] = np.where(df.ruta.str.contains('segment', case=False), 'segmentada', 'original')
df['subtipo'] = df.ruta.map(lambda p: os.path.basename(os.path.dirname(p)))
df['clase'] = np.where(df.subtipo.str.contains('benign', case=False), 'no patológica', 'patológica')
df['archivo'] = df.ruta.map(os.path.basename)

org = df[df.tipo == 'original'].reset_index(drop=True)
seg = df[df.tipo == 'segmentada'].set_index(['subtipo', 'archivo']).ruta
org['ruta_seg'] = org.set_index(['subtipo', 'archivo']).index.map(seg.get)
org = org[org.ruta_seg.notna()].reset_index(drop=True)      # nos quedamos sólo con las que tienen referencia
leer = lambda ruta: cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)

In [ ]:
TAM_EE = 1 / 3

def estimar_campo(v):
    """Campo de iluminación del canal V por cierre morfológico (justificado y comparado en el Módulo 1)."""
    n = max(v.shape)
    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(n * TAM_EE) | 1,) * 2)
    return cv2.GaussianBlur(cv2.morphologyEx(v, cv2.MORPH_CLOSE, ee), (0, 0), n / 16)

def preprocesar(rgb):
    """Corrige la iluminación en V (método 'cierre', elegido en el Módulo 1); H y S quedan intactos."""
    h, s, v = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV))
    campo = np.maximum(estimar_campo(v.astype(np.float32)), 1)
    v = np.clip(v / campo * campo.mean(), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2RGB)

## 1. Región de interés: la célula completa
La imagen segmentada de referencia tiene el fondo en negro: con un umbral bajo de intensidad alcanza para recuperar la silueta de la célula (núcleo + citoplasma).

In [ ]:
def mascara_celula(ruta_seg):
    """ROI = célula completa, a partir de la segmentación de referencia (fondo negro)."""
    g = cv2.cvtColor(leer(ruta_seg), cv2.COLOR_RGB2GRAY)
    return g > 10

ej = org.sample(1, random_state=3).iloc[0]
rgb, roi = preprocesar(leer(ej.ruta)), mascara_celula(ej.ruta_seg)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(rgb); ax[0].set_title('Original (corregida)')
ax[1].imshow(roi, cmap='gray'); ax[1].set_title('ROI (célula)')
ax[2].imshow(rgb); ax[2].imshow(np.where(roi, 1, np.nan), cmap='spring', alpha=0.4); ax[2].set_title('Superpuesta')
for a in ax: a.axis('off')
print(f"Fracción de la imagen ocupada por la célula: media = {np.mean([mascara_celula(r).mean() for r in org.ruta_seg.sample(200, random_state=0)]):.1%}")

## 2. Segmentación del núcleo
### 2.1 Umbral en S (Otsu)
Dentro de la ROI, Otsu busca el umbral de saturación que mejor separa dos poblaciones de píxeles: el núcleo (alta S) del citoplasma/eritrocitos atrapados (baja-media S). Se limpia con apertura morfológica (quita puntos sueltos) y nos quedamos sólo con componentes conexas de área razonable (descarta ruido residual).

In [ ]:
AREA_MIN = 25     # px²: componentes más chicas que esto se descartan como ruido

def limpiar(mask, area_min=AREA_MIN):
    """Apertura morfológica + descarte de componentes conexas chicas."""
    m = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, etq, stats, _ = cv2.connectedComponentsWithStats(m)
    grandes = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= area_min]
    return np.isin(etq, grandes)

def segmentar_hsv(rgb, roi):
    """Núcleo = alta saturación dentro de la ROI (umbral de Otsu calculado sólo con esos píxeles)."""
    s = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 1]
    umbral, _ = cv2.threshold(s[roi], 0, 255, cv2.THRESH_OTSU)
    return limpiar((s > umbral) & roi)

### 2.2 *k*-means en CIE Lab
Agrupamos los píxeles de la ROI en 2 clusters usando sólo **a\*** y **b\*** (color, sin luminancia: así no se confunde con el problema de iluminación ya corregido en el Módulo 1). El cluster de mayor **croma** $\sqrt{a^{*2}+b^{*2}}$ —color más "puro"/saturado— es el núcleo, consistente con que la tinción lo vuelve violeta intenso.

In [ ]:
def segmentar_kmeans(rgb, roi, k=2):
    """k-means en (a*, b*) restringido a la ROI; el cluster de mayor croma es el núcleo."""
    ab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)[..., 1:][roi] - 128
    _, etq, centros = cv2.kmeans(ab, k, None, (cv2.TERM_CRITERIA_EPS, 20, 0.5), 3, cv2.KMEANS_PP_CENTERS)
    nucleo = np.argmax(np.linalg.norm(centros, axis=1))
    mask = np.zeros(roi.shape, bool); mask[roi] = etq.ravel() == nucleo
    return limpiar(mask)

### 2.3 Comparación visual
Mismo ejemplo: máscara por cada método y su solapamiento (verde = sólo HSV, magenta = sólo *k*-means, blanco = coinciden).

In [ ]:
m_hsv, m_km = segmentar_hsv(rgb, roi), segmentar_kmeans(rgb, roi)
solape = np.zeros((*roi.shape, 3), np.uint8)
solape[m_hsv & ~m_km] = (0, 255, 0); solape[m_km & ~m_hsv] = (255, 0, 255); solape[m_hsv & m_km] = (255, 255, 255)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(m_hsv, cmap='gray'); ax[0].set_title('HSV (Otsu en S)')
ax[1].imshow(m_km, cmap='gray'); ax[1].set_title('k-means (Lab)')
ax[2].imshow(solape); ax[2].set_title('Solapamiento')
for a in ax: a.axis('off')

## 3. Comparación cuantitativa
Sobre una muestra estratificada por subtipo (al no tener máscara de referencia del núcleo, medimos qué tan de acuerdo están los dos métodos entre sí): **Dice**, el **área relativa** del núcleo (núcleo / célula) por método, y el **tiempo** de cómputo.

In [ ]:
def dice(a, b):
    """Coeficiente de Dice entre dos máscaras binarias."""
    return 2 * (a & b).sum() / (a.sum() + b.sum() + 1e-9)

muestra = org.groupby('subtipo').sample(40, random_state=0)
filas = []
for r in muestra.itertuples():
    rgb, roi = preprocesar(leer(r.ruta)), mascara_celula(r.ruta_seg)
    m1, m2 = segmentar_hsv(rgb, roi), segmentar_kmeans(rgb, roi)
    filas.append(dict(subtipo=r.subtipo, dice=dice(m1, m2),
                      area_hsv=m1.sum() / roi.sum(), area_kmeans=m2.sum() / roi.sum()))
comp = pd.DataFrame(filas)
comp.groupby('subtipo')[['dice', 'area_hsv', 'area_kmeans']].agg(['mean', 'std']).round(3)

In [ ]:
import time
t0 = time.perf_counter(); [segmentar_hsv(rgb, roi) for _ in range(20)]; t_hsv = (time.perf_counter() - t0) / 20
t0 = time.perf_counter(); [segmentar_kmeans(rgb, roi) for _ in range(20)]; t_km = (time.perf_counter() - t0) / 20
print(f"Tiempo por imagen -> HSV: {t_hsv*1000:.1f} ms · k-means: {t_km*1000:.1f} ms")

## 4. Conclusiones del Módulo 2
*(Completar con los resultados de la corrida sobre el dataset completo — ver el cuadro de arriba.)*

- Concordancia (Dice) entre HSV y *k*-means: **a completar**. Si es alta en todos los subtipos, cualquiera de los dos sirve y conviene el más rápido; si baja en algún subtipo (candidato: Pro-B, la tinción más pálida del Módulo 1 — S ≈ 18), ahí hay que mirar con más cuidado cuál se acerca más al núcleo real.
- Área relativa del núcleo: **a completar** — permite detectar si algún método sobre- o sub-segmenta de forma sistemática.
- **Decisión:** se elige el método **a completar**, por **a completar** (precisión / velocidad / robustez entre subtipos).

In [ ]:
METODO_SEGMENTACION = 'hsv'    # 'hsv' o 'kmeans' — ver conclusiones

def segmentar(rgb, roi, metodo=METODO_SEGMENTACION):
    """Segmenta el núcleo con el método elegido. Entrada: imagen ya corregida (preprocesar) y ROI (mascara_celula)."""
    return segmentar_hsv(rgb, roi) if metodo == 'hsv' else segmentar_kmeans(rgb, roi)

comp.to_csv('modulo2_comparacion_segmentacion.csv', index=False)